# Interactive Chatbot Using a Pre-trained Transformer-based LLM
**AIM :**
To develop an interactive chatbot using a pre-trained Transformer-based Large Language Model (LLM) and deploy it using Gradio.


## PART A - Loading and Configuring a Pre-trained LLM

In [12]:
%pip -q install "transformers>=4.45,<5" "gradio>=5,<7" "accelerate>=0.33" "safetensors>=0.4"

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
import time
import json
import torch
import transformers
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Gradio:", gr.__version__)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Note: CPU generation is possible but slower. Enable Kaggle GPU if available.")

Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda
PyTorch: 2.11.0+cu130
Transformers: 4.57.6
Gradio: 6.17.3
GPU: Tesla T4


In [14]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    low_cpu_mem_usage=True,
).to(DEVICE)
model.eval()

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer and model loaded successfully!")
print("Number of parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

Tokenizer and model loaded successfully!
Number of parameters: 494,032,768


In [15]:
sample_prompt = "Explain artificial intelligence in simple words."
sample_tokens = tokenizer(sample_prompt, return_tensors="pt")
print("Prompt:", sample_prompt)
print("Token IDs:", sample_tokens["input_ids"][0].tolist())
print("Tokens:", tokenizer.convert_ids_to_tokens(sample_tokens["input_ids"][0]))
print("Token count:", sample_tokens["input_ids"].shape[1])

Prompt: Explain artificial intelligence in simple words.
Token IDs: [840, 20772, 20443, 11229, 304, 4285, 4244, 13]
Tokens: ['Ex', 'plain', 'Ġartificial', 'Ġintelligence', 'Ġin', 'Ġsimple', 'Ġwords', '.']
Token count: 8


In [16]:
SYSTEM_PROMPT = (
    "You are a helpful educational chatbot. Give accurate, clear, concise answers. "
    "If unsure, say you are uncertain rather than inventing facts."
)

def generate_response(message, history=None, temperature=0.7, max_new_tokens=128):
    """Generate a response using the prompt and recent user/assistant conversation."""
    message = str(message).strip()
    if not message:
        return "Please type a question."

    history = history or []
    valid_history = [
        {"role": item["role"], "content": item["content"]}
        for item in history[-6:]
        if isinstance(item, dict)
        and item.get("role") in ("user", "assistant")
        and isinstance(item.get("content"), str)
    ]
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + valid_history
        + [{"role": "user", "content": message}]
    )
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        truncation=True,
        max_length=1536,
    ).to(DEVICE)

    use_sampling = float(temperature) > 0
    kwargs = {
        "max_new_tokens": int(max_new_tokens),
        "do_sample": use_sampling,
        "pad_token_id": tokenizer.pad_token_id,
    }
    if use_sampling:
        kwargs.update(temperature=float(temperature), top_p=0.9)

    with torch.inference_mode():
        output_ids = model.generate(**inputs, **kwargs)
    new_ids = output_ids[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

In [17]:
question = "What is deep learning? Answer in two sentences."
for temp in (0.0, 0.7, 1.0):
    start = time.perf_counter()
    answer = generate_response(question, temperature=temp, max_new_tokens=80)
    print(f"\nTemperature = {temp} | Time = {time.perf_counter()-start:.2f}s")
    print(answer)


Temperature = 0.0 | Time = 1.29s
Deep learning is a subset of machine learning that involves training artificial neural networks to perform tasks such as image recognition, speech processing, and natural language understanding by allowing the network to learn complex patterns from large datasets.

Temperature = 0.7 | Time = 0.99s
Deep learning is a subset of machine learning that uses neural networks to learn and improve their performance by processing large amounts of data through multiple layers of artificial neurons.

Temperature = 1.0 | Time = 0.76s
Deep learning is a subset of machine learning that uses specialized neural networks to model complex data structures and patterns within large datasets.


## PART B - Implementing Chatbot Functionality


In [18]:
chat_history = []
questions = [
    "My favorite subject is machine learning.",
    "What is my favorite subject?",
]
for q in questions:
    a = generate_response(q, chat_history, temperature=0.3, max_new_tokens=70)
    chat_history.extend([
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ])
    print("USER:", q)
    print("BOT:", a, "\n")

USER: My favorite subject is machine learning.
BOT: Machine learning is indeed one of the most popular and fascinating subjects in recent years. It involves teaching computers to learn from data without being explicitly programmed. This process allows machines to improve their performance over time through experience.

Some key aspects of machine learning include:

1. **Supervised Learning**: When given labeled data, the model learns patterns that can be used 

USER: What is my favorite subject?
BOT: As an AI language model, I don't have personal preferences or favorites like humans do. However, I'm designed to provide information and answer questions based on vast amounts of text data. If you have any specific topics or areas you'd like help with, feel free to ask! 



## PART C - Developing an Interactive Interface

In [19]:
def chatbot_reply(message, history, temperature, max_tokens):
    return generate_response(message, history, temperature, max_tokens)

chat_app = gr.ChatInterface(
    fn=chatbot_reply,
    title="🤖 Transformer LLM Chatbot",
    description="Chat with a pre-trained Qwen2.5-0.5B model. Built using Hugging Face Transformers and Gradio.",
    additional_inputs=[
        gr.Slider(minimum=0, maximum=1.5, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(minimum=32, maximum=256, value=128, step=32, label="Maximum new tokens"),
    ],
    examples=[
        ["What is artificial intelligence?", 0.7, 128],
        ["Explain the Transformer architecture in simple words.", 0.5, 128],
        ["Give three applications of machine learning.", 0.7, 128],
    ],
)

# share=False avoids requiring an external public Gradio tunnel.
# Set share=True ONLY if you intentionally want a temporary public link and
# Kaggle networking allows it.
chat_app.launch(
    share=True,inline=False,debug=True
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://add3101f3251a021fc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://add3101f3251a021fc.gradio.live


## PART D - Application Testing and Deployment

In [20]:
test_prompts = [
    "What is machine learning?",
    "Give two differences between AI and deep learning.",
    "Write a short motivational message for students.",
]
test_results = []
for prompt in test_prompts:
    start = time.perf_counter()
    response = generate_response(prompt, temperature=0.4, max_new_tokens=100)
    latency = round(time.perf_counter() - start, 2)
    test_results.append({"prompt": prompt, "response": response, "latency_seconds": latency})
    print("PROMPT:", prompt)
    print("RESPONSE:", response)
    print("LATENCY:", latency, "seconds\n")
print("Average response latency:", round(sum(r["latency_seconds"] for r in test_results)/len(test_results), 2), "seconds")

PROMPT: What is machine learning?
RESPONSE: Machine learning is a subset of artificial intelligence that involves the development of algorithms and statistical models that enable computers to learn from data without being explicitly programmed. In other words, it's about making systems "learn" how to perform tasks by analyzing patterns in large datasets.

Key characteristics include:

1. **Learning**: The system can automatically improve its performance over time based on new data.
2. **Prediction**: It can make predictions or decisions based on historical data.
3. **Generalization**: The model can
LATENCY: 3.03 seconds

PROMPT: Give two differences between AI and deep learning.
RESPONSE: Certainly! Here are the key differences between Artificial Intelligence (AI) and Deep Learning:

1. **Definition**:
   - **Artificial Intelligence**: This term refers to the ability of machines to perform tasks that typically require human intelligence, such as visual perception, speech recognition, d

In [21]:
# from pathlib import Path
# project_dir = Path("/kaggle/working/llm_chatbot") if Path("/kaggle/working").exists() else Path("llm_chatbot")
# project_dir.mkdir(parents=True, exist_ok=True)

# app_code = r'''import torch
# import gradio as gr
# from transformers import AutoTokenizer, AutoModelForCausalLM

# MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# SYSTEM_PROMPT = "You are a helpful educational chatbot. Be clear and do not invent facts."

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
#     low_cpu_mem_usage=True,
# ).to(DEVICE)
# model.eval()
# if tokenizer.pad_token_id is None:
#     tokenizer.pad_token = tokenizer.eos_token

# def respond(message, history, temperature, max_tokens):
#     message = message.strip()
#     if not message:
#         return "Please type a question."
#     history = history or []
#     recent = [
#         {"role": m["role"], "content": m["content"]}
#         for m in history[-6:]
#         if isinstance(m, dict)
#         and m.get("role") in ("user", "assistant")
#         and isinstance(m.get("content"), str)
#     ]
#     messages = [{"role": "system", "content": SYSTEM_PROMPT}] + recent + [{"role": "user", "content": message}]
#     inputs = tokenizer.apply_chat_template(
#         messages, add_generation_prompt=True, tokenize=True,
#         return_dict=True, return_tensors="pt", truncation=True, max_length=1536
#     ).to(DEVICE)
#     sample = float(temperature) > 0
#     kwargs = {"max_new_tokens": int(max_tokens), "do_sample": sample, "pad_token_id": tokenizer.pad_token_id}
#     if sample:
#         kwargs.update(temperature=float(temperature), top_p=0.9)
#     with torch.inference_mode():
#         output = model.generate(**inputs, **kwargs)
#     return tokenizer.decode(output[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

# demo = gr.ChatInterface(
#     fn=respond,
#     title="Transformer LLM Chatbot",
#     description="A chatbot built with a pre-trained Transformer LLM and Gradio.",
#     additional_inputs=[
#         gr.Slider(0, 1.5, value=0.7, step=0.1, label="Temperature"),
#         gr.Slider(32, 256, value=128, step=32, label="Maximum new tokens"),
#     ],
#     examples=[["What is machine learning?", 0.7, 128]],
# )
# if __name__ == "__main__":
#     demo.launch()
# '''
# (project_dir / "app.py").write_text(app_code, encoding="utf-8")
# (project_dir / "requirements.txt").write_text(
#     "torch\ntransformers>=4.45,<5\ngradio>=5,<7\naccelerate>=0.33\nsafetensors>=0.4\n", encoding="utf-8"
# )
# (project_dir / "README.md").write_text(
#     "---\ntitle: Transformer LLM Chatbot\nemoji: 🤖\ncolorFrom: blue\ncolorTo: indigo\nsdk: gradio\napp_file: app.py\n---\n\n# Transformer LLM Chatbot\n\nBuilt using Qwen2.5-0.5B-Instruct and Gradio.\n\nNote: CPU-based free Spaces may respond slowly.\n", encoding="utf-8"
# )
# print("Created:")
# for file in project_dir.iterdir():
#     print(" -", file)


### RESULT
A pre-trained Transformer-based LLM was loaded and configured. A multi-turn chatbot function and Gradio interface were implemented, response quality and latency tests were provided, and project files were prepared for deployment. Actual test outputs and successful deployment depend on execution in the user's environment.